# 00 — Setup

Confirms three things before the demo can run:

1. **Spark / Databricks Connect** session is live (or you're inside a workspace where `spark` already exists)
2. **Unity Catalog target schemas** exist — creates `{UC_SCHEMA}_bronze` / `_silver`
3. The workspace supports **VARIANT** (DBR 15.3+)

Run this first. Every other notebook expects the schemas to exist.

In [0]:
%pip install dotenv

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import os
from dotenv import load_dotenv
load_dotenv()

# Dual-mode Spark session — works locally via Databricks Connect AND inside a
# Databricks workspace notebook. In the workspace, `spark` already exists.
try:
    spark
except NameError:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless().getOrCreate()

from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

In [0]:
UC_CATALOG        = os.getenv('UC_CATALOG', 'alexander_booth')
UC_SCHEMA         = os.getenv('UC_SCHEMA',  'hockey_schema_monitoring')
BRONZE_SCHEMA     = f'{UC_SCHEMA}_bronze'
SILVER_SCHEMA     = f'{UC_SCHEMA}_silver'
MONITORING_SCHEMA = os.getenv('MONITORING_SCHEMA') or f'{SILVER_SCHEMA}_monitoring'
ALERT_EMAIL       = os.getenv('ALERT_EMAIL', 'alexander.booth@databricks.com')

BRONZE_TABLE  = f'{UC_CATALOG}.{BRONZE_SCHEMA}.plays_raw'
SILVER_TABLE  = f'{UC_CATALOG}.{SILVER_SCHEMA}.plays'
DRIFT_VIEW    = f'{UC_CATALOG}.{SILVER_SCHEMA}.v_unknown_payload_keys'

print(f'Catalog       : {UC_CATALOG}')
print(f'Bronze table  : {BRONZE_TABLE}')
print(f'Silver table  : {SILVER_TABLE}')
print(f'Drift view    : {DRIFT_VIEW}')
print(f'Monitoring    : {UC_CATALOG}.{MONITORING_SCHEMA}')
print(f'Alert email   : {ALERT_EMAIL}')

Catalog       : alexander_booth
Bronze table  : alexander_booth.hockey_schema_monitoring_bronze.plays_raw
Silver table  : alexander_booth.hockey_schema_monitoring_silver.plays
Drift view    : alexander_booth.hockey_schema_monitoring_silver.v_unknown_payload_keys
Monitoring    : alexander_booth.hockey_schema_monitoring_silver_monitoring
Alert email   : alexander.booth@databricks.com


In [0]:
spark.sql('SELECT current_user(), current_timestamp(), current_catalog()').show(truncate=False)
print('Spark version:', spark.version)

+------------------------------+--------------------------+-----------------+
|current_user()                |current_timestamp()       |current_catalog()|
+------------------------------+--------------------------+-----------------+
|alexander.booth@databricks.com|2026-05-20 13:41:44.071637|main             |
+------------------------------+--------------------------+-----------------+

Spark version: 4.1.0


In [0]:
for schema in [BRONZE_SCHEMA, SILVER_SCHEMA, MONITORING_SCHEMA]:
    spark.sql(f'CREATE SCHEMA IF NOT EXISTS {UC_CATALOG}.{schema}')
    print(f'  schema ready: {UC_CATALOG}.{schema}')

  schema ready: alexander_booth.hockey_schema_monitoring_bronze
  schema ready: alexander_booth.hockey_schema_monitoring_silver
  schema ready: alexander_booth.hockey_schema_monitoring_silver_monitoring


In [ ]:
# Confirm VARIANT is supported in this workspace (DBR 15.3+).
probe = spark.sql("SELECT typeof(parse_json('{\"k\": 1}')) AS t").collect()
assert probe[0]['t'] == 'variant', f"expected 'variant', got {probe[0]['t']!r}"
print('VARIANT probe ok — typeof =', probe[0]['t'])

## Optional reset

Uncomment to wipe the demo schemas. Useful between dry runs.

In [0]:
# for schema in [BRONZE_SCHEMA, SILVER_SCHEMA, MONITORING_SCHEMA]:
#     spark.sql(f'DROP SCHEMA IF EXISTS {UC_CATALOG}.{schema} CASCADE')
#     spark.sql(f'CREATE SCHEMA IF NOT EXISTS {UC_CATALOG}.{schema}')
# print('Schemas reset.')